# Nepal Earthquake Damage Prediction
## 03 · Model Training, Evaluation and Experimental Analysis

This notebook is the modeling stage of the group project. It assumes that:

1. `01_Preprocessing_EDA.ipynb` has produced the base engineered feature tables.
2. `02_Supplementary_Preprocessing_Geography_EDA.ipynb` has produced the complete feature tables and preserved the fixed train/validation split.

### Main research questions

1. How accurately can earthquake damage grade be predicted?
2. Which model family performs best on the fixed validation set?
3. Do the supplementary features from Notebook 02 improve predictive performance?
4. How much predictive information comes from geography versus building structure?
5. How robust is the model when evaluated on completely unseen geographic regions?
6. Can predicted Grade-3 risk support post-earthquake inspection prioritization?

### Evaluation policy

The official competition metric is micro F1. For single-label multiclass classification, micro F1 equals accuracy. We therefore also report Macro F1, Grade-3 Recall, ordinal MAE, Quadratic Weighted Kappa, and Catastrophic Error Rate.


In [3]:
%pip install lightgbm xgboost shap

Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    f1_score,
    recall_score,
    mean_absolute_error,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

RANDOM_STATE = 42

# Locate repository root whether this notebook is launched from group_project/
# or group_project/Modeling/.
CWD = Path.cwd()
if (CWD / "Preprocess & EDA").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "Preprocess & EDA").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing 'Preprocess & EDA'."
    )

PREP_DIR = PROJECT_ROOT / "Preprocess & EDA"
RAW_DIR = PREP_DIR / "raw_data"
BASE_DIR = PREP_DIR / "processed_data"
SUPP_DIR = PREP_DIR / "supplementary_processed_data"

MODELING_DIR = PROJECT_ROOT / "Modeling"
RESULT_DIR = MODELING_DIR / "results"
FIG_DIR = MODELING_DIR / "figures"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("SUPP_DIR:", SUPP_DIR.resolve())


PROJECT_ROOT: C:\Users\Welkin\OneDrive - HKUST Connect\Desktop\NTU_academic\semester1\6810\group_project
SUPP_DIR: C:\Users\Welkin\OneDrive - HKUST Connect\Desktop\NTU_academic\semester1\6810\group_project\Preprocess & EDA\supplementary_processed_data


## 1. Load final features and reproduce the fixed split

The model uses the `.pkl` feature tables from Notebook 02 because they preserve categorical dtypes. The train/validation assignment is inherited directly from preprocessing. No new random split is created here.


In [5]:
required = [
    SUPP_DIR / "train_features_complete.pkl",
    SUPP_DIR / "test_features_complete.pkl",
    SUPP_DIR / "train_validation_split_complete.csv",
    SUPP_DIR / "feature_schema.json",
    RAW_DIR / "train_labels.csv",
    RAW_DIR / "submission_format.csv",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing modeling inputs:\n" + "\n".join(missing))

train_features = pd.read_pickle(SUPP_DIR / "train_features_complete.pkl")
test_features = pd.read_pickle(SUPP_DIR / "test_features_complete.pkl")
labels = pd.read_csv(RAW_DIR / "train_labels.csv")
split_map = pd.read_csv(SUPP_DIR / "train_validation_split_complete.csv")

with open(SUPP_DIR / "feature_schema.json", "r", encoding="utf-8") as f:
    feature_schema = json.load(f)

label_map = labels.set_index("building_id")["damage_grade"]
split_lookup = split_map.set_index("building_id")["split"]

train_mask = train_features["building_id"].map(split_lookup).eq("train")
valid_mask = train_features["building_id"].map(split_lookup).eq("validation")

X_all = train_features.drop(columns="building_id").copy()
y_all = train_features["building_id"].map(label_map).astype(int)

X_train = X_all.loc[train_mask].copy()
X_valid = X_all.loc[valid_mask].copy()
y_train = y_all.loc[train_mask].copy()
y_valid = y_all.loc[valid_mask].copy()

valid_ids = train_features.loc[valid_mask, "building_id"].copy()

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))
print("Modeling features:", X_train.shape[1])
display(y_train.value_counts(normalize=True).sort_index().rename("train proportion"))
display(y_valid.value_counts(normalize=True).sort_index().rename("validation proportion"))


Training rows: 208480
Validation rows: 52121
Modeling features: 48


building_id
1    0.096407
2    0.568913
3    0.334680
Name: train proportion, dtype: float64

building_id
1    0.096410
2    0.568907
3    0.334683
Name: validation proportion, dtype: float64

## 2. Evaluation utilities

`CER` is the catastrophic error rate:

\[
CER = P(|y - \hat{y}| = 2)
\]

It counts Grade 1 → Grade 3 and Grade 3 → Grade 1 errors.


In [6]:
def catastrophic_error_rate(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)) == 2))

def evaluate_predictions(y_true, y_pred, model_name, elapsed_seconds=np.nan):
    return {
        "Model": model_name,
        "Micro F1": f1_score(y_true, y_pred, average="micro"),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Grade 3 Recall": recall_score(
            y_true, y_pred, labels=[3], average=None, zero_division=0
        )[0],
        "Ordinal MAE": mean_absolute_error(y_true, y_pred),
        "QWK": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
        "CER": catastrophic_error_rate(y_true, y_pred),
        "Seconds": elapsed_seconds,
    }

categorical_cols = [
    c for c in X_train.columns
    if str(X_train[c].dtype) in ("object", "category")
]
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

print("Categorical features:", len(categorical_cols))
print("Numeric features:", len(numeric_cols))


Categorical features: 13
Numeric features: 35


## 3. Majority baseline

A model that always predicts Grade 2 establishes the minimum benchmark.


In [7]:
results = []

dummy = DummyClassifier(strategy="most_frequent")
t0 = time.time()
dummy.fit(np.zeros((len(y_train), 1)), y_train)
pred_dummy = dummy.predict(np.zeros((len(y_valid), 1)))
results.append(
    evaluate_predictions(
        y_valid, pred_dummy, "Majority Baseline", time.time() - t0
    )
)

pd.DataFrame(results)


,Model,Micro F1,Macro F1,Grade 3 Recall,Ordinal MAE,QWK,CER,Seconds
0,Majority Baseline,0.568907,0.241742,0.0,0.431093,0.0,0.0,0.022178


## 4. Linear logistic baseline

To keep the high-cardinality categorical problem computationally manageable, we use an SGD logistic-loss classifier after one-hot encoding. This gives a linear probabilistic baseline.


In [8]:
linear_preprocessor = ColumnTransformer([
    (
        "cat",
        OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=10,
            dtype=np.float32,
        ),
        categorical_cols,
    ),
    ("num", StandardScaler(), numeric_cols),
])

linear_model = Pipeline([
    ("preprocess", linear_preprocessor),
    (
        "model",
        SGDClassifier(
            loss="log_loss",
            alpha=1e-5,
            max_iter=1000,
            tol=1e-3,
            random_state=RANDOM_STATE,
        ),
    ),
])

t0 = time.time()
linear_model.fit(X_train, y_train)
pred_linear = linear_model.predict(X_valid)
elapsed = time.time() - t0

results.append(
    evaluate_predictions(
        y_valid, pred_linear, "Linear Logistic Baseline", elapsed
    )
)

pd.DataFrame(results).sort_values("Micro F1", ascending=False)


,Model,Micro F1,Macro F1,Grade 3 Recall,Ordinal MAE,QWK,CER,Seconds
1,Linear Logistic Baseline,0.736344,0.690073,0.608633,0.270102,0.607192,0.006447,27.514535
0,Majority Baseline,0.568907,0.241742,0.000000,0.431093,0.000000,0.000000,0.022178


## 5. Compact categorical encoding for tree baselines

For Decision Tree, Random Forest, and the compact LightGBM interpretation model, category maps are fitted **only on the training split**. Unseen validation categories are mapped to `-1`.

This avoids using validation labels and avoids learning category mappings from validation rows.


In [9]:
def compact_encode_train_valid(X_train_raw, X_valid_raw, categorical):
    train_encoded = X_train_raw.copy()
    valid_encoded = X_valid_raw.copy()

    for col in categorical:
        train_values = train_encoded[col].astype(str)
        categories = pd.Index(train_values.unique())
        mapping = {value: i for i, value in enumerate(categories)}

        train_encoded[col] = train_values.map(mapping).astype("int32")
        valid_encoded[col] = (
            valid_encoded[col]
            .astype(str)
            .map(mapping)
            .fillna(-1)
            .astype("int32")
        )

    return train_encoded, valid_encoded

X_train_compact, X_valid_compact = compact_encode_train_valid(
    X_train, X_valid, categorical_cols
)


### 5.1 Decision Tree

In [10]:
tree_model = DecisionTreeClassifier(
    max_depth=18,
    min_samples_leaf=8,
    random_state=RANDOM_STATE,
)

t0 = time.time()
tree_model.fit(X_train_compact, y_train)
pred_tree = tree_model.predict(X_valid_compact)
elapsed = time.time() - t0

results.append(
    evaluate_predictions(y_valid, pred_tree, "Decision Tree", elapsed)
)

pd.DataFrame(results).sort_values("Micro F1", ascending=False)


,Model,Micro F1,Macro F1,Grade 3 Recall,Ordinal MAE,QWK,CER,Seconds
1,Linear Logistic Baseline,0.736344,0.690073,0.608633,0.270102,0.607192,0.006447,27.514535
2,Decision Tree,0.708735,0.652940,0.606914,0.297500,0.560917,0.006235,4.108625
0,Majority Baseline,0.568907,0.241742,0.000000,0.431093,0.000000,0.000000,0.022178


### 5.2 Random Forest

In [11]:
rf_model = RandomForestClassifier(
    n_estimators=80,
    max_depth=24,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

t0 = time.time()
rf_model.fit(X_train_compact, y_train)
pred_rf = rf_model.predict(X_valid_compact)
elapsed = time.time() - t0

results.append(
    evaluate_predictions(y_valid, pred_rf, "Random Forest", elapsed)
)

pd.DataFrame(results).sort_values("Micro F1", ascending=False)


,Model,Micro F1,Macro F1,Grade 3 Recall,Ordinal MAE,QWK,CER,Seconds
1,Linear Logistic Baseline,0.736344,0.690073,0.608633,0.270102,0.607192,0.006447,27.514535
3,Random Forest,0.717235,0.644612,0.533994,0.284799,0.549531,0.002034,12.375098
2,Decision Tree,0.708735,0.652940,0.606914,0.297500,0.560917,0.006235,4.108625
0,Majority Baseline,0.568907,0.241742,0.000000,0.431093,0.000000,0.000000,0.022178


### 5.3 Compact LightGBM

This compact model is also used later for controlled ablation and SHAP analysis because it preserves the original 48-feature representation.


In [12]:
compact_lgbm = LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=400,
    learning_rate=0.06,
    num_leaves=63,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=-1,
)

t0 = time.time()
compact_lgbm.fit(X_train_compact, y_train)
pred_compact_lgbm = compact_lgbm.predict(X_valid_compact)
elapsed = time.time() - t0

results.append(
    evaluate_predictions(
        y_valid,
        pred_compact_lgbm,
        "LightGBM (compact encoding)",
        elapsed,
    )
)

pd.DataFrame(results).sort_values("Micro F1", ascending=False)


,Model,Micro F1,Macro F1,Grade 3 Recall,Ordinal MAE,QWK,CER,Seconds
4,LightGBM (compact encoding),0.738685,0.681587,0.607888,0.263886,0.599257,0.002571,15.094585
1,Linear Logistic Baseline,0.736344,0.690073,0.608633,0.270102,0.607192,0.006447,27.514535
3,Random Forest,0.717235,0.644612,0.533994,0.284799,0.549531,0.002034,12.375098
2,Decision Tree,0.708735,0.652940,0.606914,0.297500,0.560917,0.006235,4.108625
0,Majority Baseline,0.568907,0.241742,0.000000,0.431093,0.000000,0.000000,0.022178


## 6. Strong one-hot boosting models

One-hot encoding avoids imposing artificial numerical order on categorical IDs. It creates a sparse matrix with thousands of columns, but both LightGBM and XGBoost can train efficiently on this representation.


In [13]:
boost_preprocessor = ColumnTransformer([
    (
        "cat",
        OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=5,
            dtype=np.float32,
        ),
        categorical_cols,
    ),
    ("num", "passthrough", numeric_cols),
])

t0 = time.time()
X_train_ohe = boost_preprocessor.fit_transform(X_train)
X_valid_ohe = boost_preprocessor.transform(X_valid)
print("Encoded training shape:", X_train_ohe.shape)
print("Preprocessing seconds:", round(time.time() - t0, 2))


Encoded training shape: (208480, 9759)
Preprocessing seconds: 4.96


### 6.1 XGBoost + One-Hot

In [14]:
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    tree_method="hist",
    eval_metric="mlogloss",
)

t0 = time.time()
xgb_model.fit(X_train_ohe, y_train - 1)
pred_xgb = xgb_model.predict(X_valid_ohe).astype(int) + 1
elapsed = time.time() - t0

results.append(
    evaluate_predictions(
        y_valid, pred_xgb, "XGBoost + One-Hot", elapsed
    )
)

pd.DataFrame(results).sort_values("Micro F1", ascending=False)


,Model,Micro F1,Macro F1,Grade 3 Recall,Ordinal MAE,QWK,CER,Seconds
4,LightGBM (compact encoding),0.738685,0.681587,0.607888,0.263886,0.599257,0.002571,15.094585
1,Linear Logistic Baseline,0.736344,0.690073,0.608633,0.270102,0.607192,0.006447,27.514535
5,XGBoost + One-Hot,0.732641,0.669293,0.577505,0.269623,0.582896,0.002264,49.407840
3,Random Forest,0.717235,0.644612,0.533994,0.284799,0.549531,0.002034,12.375098
2,Decision Tree,0.708735,0.652940,0.606914,0.297500,0.560917,0.006235,4.108625
0,Majority Baseline,0.568907,0.241742,0.000000,0.431093,0.000000,0.000000,0.022178


### 6.2 LightGBM + One-Hot

This was the strongest validation model in our verified run.


In [15]:
best_model = LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=400,
    learning_rate=0.06,
    num_leaves=63,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=-1,
)

t0 = time.time()
best_model.fit(X_train_ohe, y_train)
pred_best = best_model.predict(X_valid_ohe)
proba_best = best_model.predict_proba(X_valid_ohe)
elapsed = time.time() - t0

results.append(
    evaluate_predictions(
        y_valid, pred_best, "LightGBM + One-Hot", elapsed
    )
)

model_results = (
    pd.DataFrame(results)
    .drop_duplicates("Model", keep="last")
    .sort_values("Micro F1", ascending=False)
    .reset_index(drop=True)
)

display(model_results)
model_results.to_csv(RESULT_DIR / "model_comparison.csv", index=False)


,Model,Micro F1,Macro F1,Grade 3 Recall,Ordinal MAE,QWK,CER,Seconds
0,LightGBM + One-Hot,0.747395,0.693652,0.622220,0.255693,0.614189,0.003089,41.812045
1,LightGBM (compact encoding),0.738685,0.681587,0.607888,0.263886,0.599257,0.002571,15.094585
2,Linear Logistic Baseline,0.736344,0.690073,0.608633,0.270102,0.607192,0.006447,27.514535
3,XGBoost + One-Hot,0.732641,0.669293,0.577505,0.269623,0.582896,0.002264,49.407840
4,Random Forest,0.717235,0.644612,0.533994,0.284799,0.549531,0.002034,12.375098
5,Decision Tree,0.708735,0.652940,0.606914,0.297500,0.560917,0.006235,4.108625
6,Majority Baseline,0.568907,0.241742,0.000000,0.431093,0.000000,0.000000,0.022178


### Verified reference results from the completed run

| Model                       |   Micro F1 |   Macro F1 |   Grade 3 Recall |   Ordinal MAE |    QWK |    CER |
|:----------------------------|-----------:|-----------:|-----------------:|--------------:|-------:|-------:|
| LightGBM + One-Hot          |     0.7474 |     0.6937 |           0.6222 |        0.2557 | 0.6142 | 0.0031 |
| LightGBM (compact encoding) |     0.7387 |     0.6816 |           0.6079 |        0.2639 | 0.5993 | 0.0026 |
| Linear Logistic Baseline    |     0.7359 |     0.6835 |           0.5488 |        0.2692 | 0.5922 | 0.0051 |
| XGBoost + One-Hot           |     0.7331 |     0.6700 |           0.5792 |        0.2692 | 0.5838 | 0.0023 |
| Random Forest               |     0.7172 |     0.6446 |           0.5340 |        0.2848 | 0.5495 | 0.0020 |
| Decision Tree               |     0.7087 |     0.6529 |           0.6069 |        0.2975 | 0.5609 | 0.0062 |
| Majority Baseline           |     0.5689 |     0.2417 |           0.0000 |        0.4311 | 0.0000 | 0.0000 |

![Model comparison](figures/01_model_comparison.png)

**Interpretation.** LightGBM with one-hot encoded categoricals achieved the strongest validation performance. The linear baseline was surprisingly competitive, indicating that much of the signal is additive, while boosting still provided a meaningful gain by modeling nonlinearities and interactions.


## 7. Best-model error analysis

In [16]:
cm = confusion_matrix(y_valid, pred_best, labels=[1, 2, 3])
cm_df = pd.DataFrame(
    cm,
    index=["Actual 1", "Actual 2", "Actual 3"],
    columns=["Pred 1", "Pred 2", "Pred 3"],
)
display(cm_df)

print(classification_report(y_valid, pred_best, digits=4))

cm_df.to_csv(RESULT_DIR / "best_confusion_matrix.csv")


,Pred 1,Pred 2,Pred 3
Actual 1,2614,2350,61
Actual 2,1057,25487,3108
Actual 3,100,6490,10854


              precision    recall  f1-score   support

           1     0.6932    0.5202    0.5944      5025
           2     0.7425    0.8595    0.7967     29652
           3     0.7740    0.6222    0.6899     17444

    accuracy                         0.7474     52121
   macro avg     0.7366    0.6673    0.6937     52121
weighted avg     0.7483    0.7474    0.7415     52121



![Confusion matrix](figures/02_confusion_matrix.png)

The best model is strongest on Grade 2. Grade 1 is the hardest minority class. For Grade 3, recall is about 0.62 and precision is about 0.77. Extreme Grade-1 ↔ Grade-3 errors are rare, producing a catastrophic error rate of roughly 0.31%.


## 8. Did Notebook 02 feature engineering improve predictive performance?

We use the same compact LightGBM configuration and the same validation split to compare:

- Notebook 01 engineered features
- Notebook 01 + Notebook 02 complete features


In [17]:
base_train = pd.read_csv(BASE_DIR / "train_features_engineered.csv")

def prepare_compact_frame(df, split_lookup, label_map):
    tr = df["building_id"].map(split_lookup).eq("train")
    va = df["building_id"].map(split_lookup).eq("validation")
    y = df["building_id"].map(label_map).astype(int)

    X = df.drop(columns="building_id").copy()
    cats = [
        c for c in X.columns
        if str(X[c].dtype) in ("object", "category")
        or c.startswith("geo_level_")
    ]

    Xt, Xv = compact_encode_train_valid(
        X.loc[tr], X.loc[va], cats
    )
    return Xt, Xv, y.loc[tr], y.loc[va]

X01_tr, X01_va, y01_tr, y01_va = prepare_compact_frame(
    base_train, split_lookup, label_map
)

ablation_model_01 = LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=350,
    learning_rate=0.07,
    num_leaves=63,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=-1,
)
ablation_model_01.fit(X01_tr, y01_tr)
pred_01 = ablation_model_01.predict(X01_va)

ablation_model_02 = LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=350,
    learning_rate=0.07,
    num_leaves=63,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=-1,
)
ablation_model_02.fit(X_train_compact, y_train)
pred_02 = ablation_model_02.predict(X_valid_compact)

feature_ablation_results = pd.DataFrame([
    evaluate_predictions(y01_va, pred_01, "Notebook 01 features"),
    evaluate_predictions(y_valid, pred_02, "Notebook 01 + 02 features"),
])
display(feature_ablation_results)


,Model,Micro F1,Macro F1,Grade 3 Recall,Ordinal MAE,QWK,CER,Seconds
0,Notebook 01 features,0.739817,0.682305,0.608404,0.262927,0.600046,0.002744,NaN
1,Notebook 01 + 02 features,0.738973,0.681006,0.608003,0.263560,0.599234,0.002533,NaN


### Verified reference result

| Feature Set               |   Micro F1 |   Macro F1 |   Grade 3 Recall |
|:--------------------------|-----------:|-----------:|-----------------:|
| Notebook 01 features      |     0.7398 |     0.6823 |           0.6084 |
| Notebook 01 + 02 features |     0.7387 |     0.6816 |           0.6079 |

![Feature ablation](figures/03_feature_engineering_ablation.png)

**Finding.** The four supplementary features did not improve this LightGBM validation score. The change is extremely small and slightly negative. Their value is therefore mainly richer representation and interpretability unless another model demonstrates a measurable benefit.

## 9. Geography versus structure

We compare three feature sets using the same compact LightGBM:

1. `Geography only`: the three hierarchical geographic IDs
2. `Structure only`: all non-geographic features
3. `Full`: all features


In [18]:
GEO_COLS = [
    "geo_level_1_id",
    "geo_level_2_id",
    "geo_level_3_id",
]

def run_feature_subset(feature_list, label):
    Xt = X_train[feature_list].copy()
    Xv = X_valid[feature_list].copy()
    cats = [
        c for c in feature_list
        if str(Xt[c].dtype) in ("object", "category")
    ]
    Xt_enc, Xv_enc = compact_encode_train_valid(Xt, Xv, cats)

    model = LGBMClassifier(
        objective="multiclass",
        num_class=3,
        n_estimators=350,
        learning_rate=0.07,
        num_leaves=63,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=-1,
    )
    model.fit(Xt_enc, y_train)
    pred = model.predict(Xv_enc)
    return evaluate_predictions(y_valid, pred, label)

structure_cols = [c for c in X_train.columns if c not in GEO_COLS]

geo_structure_results = pd.DataFrame([
    run_feature_subset(GEO_COLS, "Geography only"),
    run_feature_subset(structure_cols, "Structure only"),
    evaluate_predictions(
        y_valid,
        pred_compact_lgbm,
        "Full compact model",
    ),
])

display(geo_structure_results)


,Model,Micro F1,Macro F1,Grade 3 Recall,Ordinal MAE,QWK,CER,Seconds
0,Geography only,0.716583,0.614963,0.590117,0.291380,0.518756,0.007962,NaN
1,Structure only,0.620076,0.530967,0.308072,0.384125,0.354451,0.004202,NaN
2,Full compact model,0.738685,0.681587,0.607888,0.263886,0.599257,0.002571,NaN


### Verified reference result

| Feature Set        |   Micro F1 |   Macro F1 |   Grade 3 Recall |
|:-------------------|-----------:|-----------:|-----------------:|
| Geography only     |     0.7166 |     0.6150 |           0.5901 |
| Structure only     |     0.6201 |     0.5310 |           0.3081 |
| Full compact model |     0.7387 |     0.6816 |           0.6079 |

![Geography vs structure](figures/04_geography_vs_structure.png)

**Finding.** Geography alone is substantially stronger than structure alone. The full model still improves over geography alone, so structural variables contain incremental predictive information, but location is the dominant source of signal in the random holdout.

## 10. Geographic holdout: can the model generalize to unseen regions?

A random split allows the same broad geographic regions to appear in both training and validation. We therefore hold out entire `geo_level_1_id` groups.

This is a deliberately harder distribution-shift test. The resulting score should not be compared to the random split as if the two validation sets were identical.


In [19]:
groups = train_features["geo_level_1_id"].astype(str)
indices = np.arange(len(train_features))

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)
geo_train_idx, geo_valid_idx = next(
    gss.split(indices, y_all, groups=groups)
)

X_geo_train_raw = X_all.iloc[geo_train_idx].copy()
X_geo_valid_raw = X_all.iloc[geo_valid_idx].copy()
y_geo_train = y_all.iloc[geo_train_idx].copy()
y_geo_valid = y_all.iloc[geo_valid_idx].copy()

geo_cats = [
    c for c in X_all.columns
    if str(X_all[c].dtype) in ("object", "category")
]

X_geo_train, X_geo_valid = compact_encode_train_valid(
    X_geo_train_raw,
    X_geo_valid_raw,
    geo_cats,
)

geo_model = LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=350,
    learning_rate=0.07,
    num_leaves=63,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=-1,
)

geo_model.fit(X_geo_train, y_geo_train)
pred_geo_holdout = geo_model.predict(X_geo_valid)

geo_holdout_metrics = evaluate_predictions(
    y_geo_valid,
    pred_geo_holdout,
    "Unseen Geo-Level-1 holdout",
)
display(pd.DataFrame([geo_holdout_metrics]))

print("Training target distribution:")
display(y_geo_train.value_counts(normalize=True).sort_index())
print("Geographic-holdout target distribution:")
display(y_geo_valid.value_counts(normalize=True).sort_index())


,Model,Micro F1,Macro F1,Grade 3 Recall,Ordinal MAE,QWK,CER,Seconds
0,Unseen Geo-Level-1 holdout,0.510873,0.433471,0.917785,0.505938,0.263757,0.016811,NaN


Training target distribution:


building_id
1    0.116838
2    0.604578
3    0.278584
Name: proportion, dtype: float64

Geographic-holdout target distribution:


building_id
1    0.053985
2    0.494853
3    0.451162
Name: proportion, dtype: float64

![Geographic holdout](figures/05_random_vs_geo_holdout.png)

**Verified finding.** Micro F1 dropped to about **0.511** when seven entire Geo-Level-1 regions were held out. This is strong evidence that the random-split score is not fully transferable to unseen regions. However, the held-out regions also have a very different target distribution, with Grade 3 accounting for about 45% of observations versus about 28% in the geographic-holdout training portion. Therefore the entire drop cannot be attributed purely to geographic memorization; it reflects a combination of unseen geography and distribution shift.

## 11. Explainability with a compact LightGBM SHAP model

For readable global explanations, we use the compact 48-feature LightGBM representation. It scores slightly below the one-hot winner, but preserves the original feature names and makes SHAP aggregation straightforward.


In [20]:
import shap

sample = X_valid_compact.sample(
    n=min(1200, len(X_valid_compact)),
    random_state=RANDOM_STATE,
)

explainer = shap.TreeExplainer(compact_lgbm)
shap_values = explainer.shap_values(sample)

arr = np.asarray(shap_values)

if arr.ndim == 3 and arr.shape[0] == len(sample):
    mean_abs_shap = np.abs(arr).mean(axis=(0, 2))
elif arr.ndim == 3 and arr.shape[1] == len(sample):
    mean_abs_shap = np.abs(arr).mean(axis=(0, 1))
elif isinstance(shap_values, list):
    mean_abs_shap = np.mean(
        [np.abs(np.asarray(v)).mean(axis=0) for v in shap_values],
        axis=0,
    )
else:
    mean_abs_shap = np.abs(arr).mean(axis=0)

shap_importance = (
    pd.DataFrame({
        "feature": X_train_compact.columns,
        "mean_abs_shap": mean_abs_shap,
    })
    .sort_values("mean_abs_shap", ascending=False)
)

display(shap_importance.head(15))
shap_importance.to_csv(
    RESULT_DIR / "compact_model_shap_importance.csv",
    index=False,
)


,feature,mean_abs_shap
0,geo_level_1_id,0.534034
1,geo_level_2_id,0.150329
4,age,0.139168
8,foundation_type,0.134256
15,has_superstructure_mud_mortar_stone,0.108016
11,other_floor_type,0.065987
10,ground_floor_type,0.061552
2,geo_level_3_id,0.052337
9,roof_type,0.045757
3,count_floors_pre_eq,0.044154


![SHAP importance](figures/06_shap_importance.png)

**Finding.** `geo_level_1_id` is the dominant global feature. The next tier includes `geo_level_2_id`, building age, foundation type, and mud-mortar-stone superstructure. The importance ranking is consistent with the preprocessing EDA: geography, structural system, age, foundation, floors, and roof all carry meaningful damage information.

## 12. From classification to inspection prioritization

We rank validation buildings by the best model's predicted probability of Grade 3. This simulates a limited inspection budget.


In [21]:
grade3_prob = proba_best[:, 2]

ranking = pd.DataFrame({
    "building_id": valid_ids.to_numpy(),
    "actual": y_valid.to_numpy(),
    "p_grade3": grade3_prob,
}).sort_values("p_grade3", ascending=False)

base_rate = (ranking["actual"] == 3).mean()

priority_rows = []
for fraction in [0.05, 0.10, 0.20, 0.30]:
    n = max(1, int(len(ranking) * fraction))
    top = ranking.head(n)

    precision = (top["actual"] == 3).mean()
    recall = (
        (top["actual"] == 3).sum()
        / (ranking["actual"] == 3).sum()
    )

    priority_rows.append({
        "Top Fraction": fraction,
        "Buildings Selected": n,
        "Precision@K": precision,
        "Recall@K": recall,
        "Lift@K": precision / base_rate,
    })

priority_results = pd.DataFrame(priority_rows)
display(priority_results)

priority_results.to_csv(
    RESULT_DIR / "inspection_prioritization.csv",
    index=False,
)


,Top Fraction,Buildings Selected,Precision@K,Recall@K,Lift@K
0,0.05,2606,0.966232,0.144348,2.887008
1,0.10,5212,0.938987,0.280555,2.805603
2,0.20,10424,0.840848,0.502465,2.512373
3,0.30,15636,0.743796,0.666705,2.222392


### Verified reference result

|   Top Fraction |   Buildings Selected |   Precision@K |   Recall@K |   Lift@K |
|---------------:|---------------------:|--------------:|-----------:|---------:|
|         0.0500 |            2606.0000 |        0.9662 |     0.1443 |   2.8870 |
|         0.1000 |            5212.0000 |        0.9390 |     0.2806 |   2.8056 |
|         0.2000 |           10424.0000 |        0.8408 |     0.5025 |   2.5124 |
|         0.3000 |           15636.0000 |        0.7438 |     0.6667 |   2.2224 |

![Inspection prioritization](figures/07_inspection_prioritization.png)

**Finding.** If only the highest-risk 10% of validation buildings were inspected first, about 93.9% of those inspected would truly be Grade 3, capturing about 28.1% of all Grade-3 buildings. At a 20% inspection budget, the model captures about 50.2% of all Grade-3 buildings while maintaining about 84.1% precision.

## 13. Final training and DrivenData submission

After model selection is complete, retrain on all labeled observations and generate predictions for the official test set.

The reference submission included with this project uses the same LightGBM + one-hot architecture with a computationally lighter 250-tree full-data fit. If runtime is acceptable on your machine, you may increase `n_estimators` to 400 to match the validation configuration.


In [22]:
X_full = train_features.drop(columns="building_id").copy()
X_test = test_features.drop(columns="building_id").copy()
y_full = train_features["building_id"].map(label_map).astype(int)

full_categorical = [
    c for c in X_full.columns
    if str(X_full[c].dtype) in ("object", "category")
]
full_numeric = [c for c in X_full.columns if c not in full_categorical]

final_preprocessor = ColumnTransformer([
    (
        "cat",
        OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=5,
            dtype=np.float32,
        ),
        full_categorical,
    ),
    ("num", "passthrough", full_numeric),
])

X_full_ohe = final_preprocessor.fit_transform(X_full)
X_test_ohe = final_preprocessor.transform(X_test)

final_model = LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=250,  # raise to 400 if desired
    learning_rate=0.08,
    num_leaves=63,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=-1,
)

final_model.fit(X_full_ohe, y_full)
test_pred = final_model.predict(X_test_ohe).astype(int)

submission = pd.DataFrame({
    "building_id": test_features["building_id"],
    "damage_grade": test_pred,
})

submission.to_csv(
    RESULT_DIR / "submission_lightgbm_ohe.csv",
    index=False,
)

display(submission.head())
display(submission["damage_grade"].value_counts(normalize=True).sort_index())


,building_id,damage_grade
0,300051,3
1,99355,2
2,890251,2
3,745817,1
4,421793,3


damage_grade
1    0.070360
2    0.660819
3    0.268822
Name: proportion, dtype: float64

## 14. Main conclusions

1. **Best random-holdout model:** LightGBM with one-hot encoded categorical variables, Micro F1 ≈ **0.7474** and Macro F1 ≈ **0.6937**.
2. **Class difficulty:** Grade 2 is easiest; Grade 1 remains the hardest class. Grade-3 recall is about **0.622**.
3. **Supplementary features:** the four Notebook-02 features do not materially improve compact-LightGBM validation performance.
4. **Geography dominates:** geography-only performance is much stronger than structure-only performance, but the full model still improves over geography alone.
5. **Transferability is weak:** performance drops sharply on completely unseen Geo-Level-1 regions. The drop also includes substantial target-distribution shift.
6. **Explainability:** geography, age, foundation, mud-mortar-stone structure, floor type, and roof type are among the strongest predictors.
7. **Operational use:** risk ranking can concentrate severe-damage buildings into the top inspection tiers, but this should be interpreted within the same-distribution validation setting.
8. **Causal caution:** feature importance and SHAP describe predictive association. They do not establish that changing a feature would causally change earthquake damage.


## 15. Leaderboard Optimization Summary

After completing the main modeling and experimental analysis, we carried out a separate leaderboard optimization stage. The purpose of this stage was to improve the DrivenData public Micro F1 score through stronger model configurations and model ensembling, while keeping the main course-project analysis unchanged.

The public leaderboard progression was:

| Version | Method | Public Micro F1 |
|---|---|---:|
| Original | LightGBM + One-Hot | 0.7440 |
| A | LightGBM, 400 trees | 0.7451 |
| B | 80% LightGBM + 20% Logistic probability blend | 0.7456 |
| C | Probability blend with mild class-score adjustment | 0.7460 |
| D | Native CatBoost | 0.7486 |
| E | LightGBM + CatBoost + Logistic ensemble | 0.7494 |
| **Final** | **Tuned CatBoost + Tuned LightGBM probability ensemble** | **0.7515** |

At the time of our latest submission, the highest public leaderboard score was **0.7564**. Our best submission achieved **0.7515**, only **0.0049** below the leading score, and ranked **252nd out of 2,761 competitors**, placing our team approximately in the **top 9.1%** of the public leaderboard.

The results show that the largest improvements came from introducing a different model family, CatBoost, and then combining tuned CatBoost and LightGBM probabilities. Small adjustments to tree count or class scores produced only limited gains.

The final leaderboard-optimization implementation is provided separately in:

`04_CatBoost_LightGBM_Tuning_Ensemble.ipynb`

Final leaderboard-optimization csv file is in:

`result_04_tuning\04_submission_tuned_lgbm_catboost`


Notebook 04 contains the tuned LightGBM and CatBoost experiments, probability-weight search, final ensemble selection, full-data retraining, and generation of the best submission file.

The leaderboard stage is treated as an optional extension. The main academic conclusions of the project are based on the validation experiments, feature ablation, geography-versus-structure analysis, geographic holdout, SHAP interpretation, and inspection-prioritization analysis presented in this notebook.